## Censo universitarios 1996 

In [33]:
# Paquetes
from pathlib import Path

import pandas as pd
import pyreadstat
import matplotlib.pyplot as plt
import seaborn as sns

In [34]:
# Rutas: Censo 1996
ruta_1996 = Path(r"E:\BD\Censo Universitario\1996")
ruta_pregrado_1996 = ruta_1996 / "1 Estudiante de Pre-Grado"
ruta_universidades_1996 = ruta_1996 / "6 Universidad"

# Cargar los cinco módulos de pregrado
bases_pregrado_1996 = {}
metadatos_pregrado_1996 = {}

for modulo in ["M1S01", "M1S02", "M1S03", "M1S04", "M1S05"]:
    datos, meta = pyreadstat.read_sav(str(ruta_pregrado_1996 / f"{modulo}.sav"))
    datos["key"] = (datos["key"].astype("string").str.strip().replace("", pd.NA))

    bases_pregrado_1996[modulo] = datos
    metadatos_pregrado_1996[modulo] = meta

# Cargar la base de universidades
universidades_1996, meta_universidades_1996 = pyreadstat.read_sav(str(ruta_universidades_1996 / "M6S01.sav"))
universidades_1996.columns = universidades_1996.columns.str.lower()

In [36]:
# Bases de pregrado
m1s01 = bases_pregrado_1996["M1S01"]
m1s02 = bases_pregrado_1996["M1S02"]
m1s03 = bases_pregrado_1996["M1S03"]
m1s04 = bases_pregrado_1996["M1S04"]
m1s05 = bases_pregrado_1996["M1S05"]

In [37]:
# Revisar duplicados en cada módulo
for nombre, base in [
    ("M1S01", m1s01),
    ("M1S02", m1s02),
    ("M1S03", m1s03),
    ("M1S04", m1s04),
    ("M1S05", m1s05),
]:
    print(f"{nombre}: {base['key'].duplicated().sum():,} duplicados")

M1S01: 14 duplicados
M1S02: 3 duplicados
M1S03: 3 duplicados
M1S04: 3 duplicados
M1S05: 3 duplicados


In [38]:
# Preparar los módulos y resolver duplicados
modulos = {}
registros_descartados = {}

for modulo, base in bases_pregrado_1996.items():
    datos = base.dropna(subset=["key"]).drop_duplicates().copy()
    datos["_completos"] = (datos.drop(columns="key").replace(r"^\s*$", pd.NA, regex=True).notna().sum(axis=1))
    datos = datos.sort_values("_completos", ascending=False, kind="stable")
    descartar = datos["key"].duplicated(keep="first")
    registros_descartados[modulo] = (datos.loc[descartar].drop(columns="_completos"))
    
    modulos[modulo] = (
        datos.loc[~descartar].drop(columns="_completos").rename(columns={columna: f"{modulo}_{columna}"
            for columna in base.columns
            if columna != "key"
        })
    )

# Conservar las coincidencias entre M1S01 y M1S02
pregrado_1996 = modulos["M1S01"].merge(modulos["M1S02"],on="key",how="inner",validate="one_to_one",)

# Incorporar los otros módulos sin reducir la población inicial
for modulo in ["M1S03", "M1S04", "M1S05"]:
    pregrado_1996 = pregrado_1996.merge(
        modulos[modulo],
        on="key",
        how="left",
        validate="one_to_one",
    )
pregrado_1996.insert(0, "id_fila", range(1, len(pregrado_1996) + 1))

print(f"Base integrada de pregrado: {len(pregrado_1996):,} registros")

Base integrada de pregrado: 335,711 registros


In [39]:
# Seleccionar las variables para preparar la base de análisis
variables_pregrado = {
    "id_fila": "id_fila",
    "key": "key",
    "M1S01_p3": "codigo_universidad",
    "M1S01_p4": "codigo_facultad",
    "M1S01_p5": "codigo_programa",
    "M1S01_p5_a": "nombre_programa",
    "M1S02_p6": "sexo",
    "M1S02_p7_1": "dia_nacimiento",
    "M1S02_p7_2": "mes_nacimiento",
    "M1S02_p7_3": "anio_nacimiento_original",
    "M1S02_p9": "codigo_nacionalidad",
    "M1S01_p2": "ubigeo_original",
    "M1S01_p2_nomdpto": "departamento_original",
    "M1S01_p2_nomprov": "provincia_original",
    "M1S01_p2_nomdist": "distrito_original",
    "M1S02_p8": "codigo_lugar_nacimiento",
}

pregrado_1996_analisis = (
    pregrado_1996[list(variables_pregrado)]
    .rename(columns=variables_pregrado)
    .copy()
)
pregrado_1996_analisis["nivel_academico"] = "Pregrado"

# Preparar el catálogo de universidades
catalogo_universidades_1996 = (
    universidades_1996[["p1_1", "p1_2", "p2"]]
    .rename(columns={
        "p1_1": "codigo_universidad",
        "p1_2": "nombre_universidad",
        "p2": "tipo_universidad",
    })
    .copy()
)

catalogo_universidades_1996["nombre_universidad"] = (
    catalogo_universidades_1996["nombre_universidad"].str.strip()
)

catalogo_universidades_1996["tipo_universidad"] = (
    catalogo_universidades_1996["tipo_universidad"].map({
        1: "Pública",
        2: "Privada",
    })
)

# Incorporar nombre y tipo de universidad
pregrado_1996_analisis = pregrado_1996_analisis.merge(
    catalogo_universidades_1996,
    on="codigo_universidad",
    how="left",
    validate="many_to_one",
)

In [40]:
pregrado_1996_analisis.head()

,id_fila,key,codigo_universidad,codigo_facultad,codigo_programa,nombre_programa,sexo,dia_nacimiento,mes_nacimiento,anio_nacimiento_original,codigo_nacionalidad,ubigeo_original,departamento_original,provincia_original,distrito_original,codigo_lugar_nacimiento,nivel_academico,nombre_universidad,tipo_universidad
0,1,1421 546,1.0,421.0,1.0,PSICOLOGIA,1.0,19.0,12.0,75.0,4028.0,150101,LIMA,LIMA,LIMA,150113.0,Pregrado,UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS,Pública
1,2,1421 550,1.0,421.0,1.0,PSICOLOGIA,1.0,0.0,0.0,0.0,4028.0,150132,LIMA,LIMA,SAN JUAN DE LURIGANCHO,190199.0,Pregrado,UNIVERSIDAD NACIONAL MAYOR DE SAN MARCOS,Pública
2,3,2412 1010,2.0,412.0,4.0,EDUCACION FISICA,2.0,16.0,10.0,72.0,4028.0,050101,AYACUCHO,HUAMANGA,AYACUCHO,50101.0,Pregrado,UNIVERSIDAD NACIONAL SAN CRISTOBAL DE HUAMANGA,Pública
3,4,2414 36,2.0,414.0,2.0,ARQUEOLOGIA E HISTORIA,1.0,7.0,10.0,72.0,4028.0,050101,AYACUCHO,HUAMANGA,AYACUCHO,51108.0,Pregrado,UNIVERSIDAD NACIONAL SAN CRISTOBAL DE HUAMANGA,Pública
4,5,2414 37,2.0,414.0,2.0,ARQUEOLOGIA E HISTORIA,2.0,24.0,10.0,68.0,4028.0,050101,AYACUCHO,HUAMANGA,AYACUCHO,50101.0,Pregrado,UNIVERSIDAD NACIONAL SAN CRISTOBAL DE HUAMANGA,Pública
